# M2 Notebook 25 — Treatment-Effect and Uplift Modeling

**Status:** Runnable first edition

## Learning objectives

- Estimate heterogeneous treatment effects.
- Compare S-learners and T-learners.
- Evaluate targeting policies.

In [ ]:
from srai_math.utils import environment_info,set_seed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()
from srai_ml import (
    SLearner,TLearner,policy_value,uplift_by_quantile,
)


In [ ]:
rng=np.random.default_rng(25)
n=3000
X=rng.normal(size=(n,3))
t=rng.binomial(1,.5,size=n)
true_tau=1+1.5*X[:,0]-.5*X[:,1]
y=2+X[:,2]+t*true_tau+rng.normal(scale=1,size=n)
tlearner=TLearner().fit(X,t,y)
slearner=SLearner().fit(X,t,y)
cate_t=tlearner.predict_cate(X)
cate_s=slearner.predict_cate(X)
{"T_learner_correlation":np.corrcoef(cate_t,true_tau)[0,1],
 "S_learner_correlation":np.corrcoef(cate_s,true_tau)[0,1]}


## Targeting policy

In [ ]:
recommended=(cate_t>0).astype(int)
{"observed_policy_value":policy_value(y,t,recommended),
 "treat_all_value":policy_value(y,t,np.ones(n,int)),
 "treat_none_value":policy_value(y,t,np.zeros(n,int))}


## Uplift by quantile

In [ ]:
pd.DataFrame(
    uplift_by_quantile(cate_t,y,t,5),
    columns=["quantile","mean_predicted_CATE","observed_uplift"]
)


## Decision Intelligence case

Treatment-effect models can support targeted intervention only when the underlying causal assumptions and policy constraints are credible.

## Key insight

Uplift modeling asks who benefits from treatment—not merely who has a high predicted outcome.